In [2]:
import pandas as pd

In [3]:
df = pd.read_csv('titanic.csv')

In [4]:
df.shape

(891, 12)

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB


In [6]:
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [9]:
missing_data = df.isnull().sum()

In [10]:
print(missing_data[missing_data > 0])

Age         177
Cabin       687
Embarked      2
dtype: int64


In [12]:
df['Has_Cabin'] = df['Cabin'].notnull().astype(int)
print(df.groupby('Has_Cabin')['Survived'].mean())

Has_Cabin
0    0.299854
1    0.666667
Name: Survived, dtype: float64


In [14]:
mode_port = df['Embarked'].mode()[0]
print(f"The most embarked port is: {mode_port}")

The most embarked port is: S


In [15]:
df['Embarked'] = df['Embarked'].fillna(mode_port)

In [17]:
#for AGE column - handling missing data

# Step 1: Extract the title from the Name column using a regular expression
df['Title'] = df['Name'].str.extract(' ([A-Za-z]+)\.', expand=False)

In [18]:
print(df.groupby('Title')['Age'].median())

Title
Capt        70.0
Col         58.0
Countess    33.0
Don         40.0
Dr          46.5
Jonkheer    38.0
Lady        48.0
Major       48.5
Master       3.5
Miss        21.0
Mlle        24.0
Mme         24.0
Mr          30.0
Mrs         35.0
Ms          28.0
Rev         46.5
Sir         49.0
Name: Age, dtype: float64


In [19]:
# Calculate the median age for each title and fill the missing values inline
df['Age'] = df['Age'].fillna(df.groupby('Title')['Age'].transform('median'))

# Verify if we have any missing values left in Age
print(f"Missing values in Age: {df['Age'].isnull().sum()}")

Missing values in Age: 0


In [20]:
df['Fare'].describe()

count    891.000000
mean      32.204208
std       49.693429
min        0.000000
25%        7.910400
50%       14.454200
75%       31.000000
max      512.329200
Name: Fare, dtype: float64

In [22]:
#Testing the hypothesis : Did money buy survival?


pclass_survival = df.groupby('Pclass')['Survived'].mean() * 100
pclass_survival

Pclass
1    62.962963
2    47.282609
3    24.236253
Name: Survived, dtype: float64

In [27]:
# Now let's check the classic rule of titanic: "Women and children first."
gender_survival = df.groupby('Sex')['Survived'].mean() * 100
gender_survival

#output shows 74% of women survived.

Sex
female    74.203822
male      18.890815
Name: Survived, dtype: float64

In [36]:
# Function to define the passenger type
def get_person_category(passenger):
    age, sex = passenger
    if age < 16:
        return 'child'
    else:
        return sex

# Apply the function across our rows
df['Person_Category'] = df[['Age', 'Sex']].apply(get_person_category, axis=1)

# Now, let's cross-tabulate this new feature with survival rates
category_survival = df.groupby('Person_Category')['Survived'].mean() * 100
print(category_survival)

Person_Category
child     58.620690
female    75.645756
male      16.135084
Name: Survived, dtype: float64


In [37]:
df['Family_Size'] = df['SibSp'] + df['Parch'] + 1
print(df.groupby('Family_Size')['Survived'].mean() * 100)

Family_Size
1     30.353818
2     55.279503
3     57.843137
4     72.413793
5     20.000000
6     13.636364
7     33.333333
8      0.000000
11     0.000000
Name: Survived, dtype: float64
